In [1]:
import time
import re
import json
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import gc
import datetime

def create_driver():
    """새 드라이버 생성"""
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.page_load_strategy = 'eager'
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    return webdriver.Chrome(options=options)

def navigate_to_page(driver, page_num):
    """특정 페이지로 이동"""
    kw = '부산%20맛집'
    url = f'https://map.naver.com/p/search/{kw}?c=9.00,0,0,0,dh'
    driver.get(url)
    time.sleep(4)
    
    # searchIframe 진입
    driver.switch_to.frame("searchIframe")
    time.sleep(2)
    
    # 원하는 페이지로 이동
    if page_num > 1:
        for p in range(2, page_num + 1):
            try:
                # 페이지 버튼 클릭
                page_btn = driver.find_element(By.XPATH, f'//a[contains(@class, "mBN2s") and normalize-space(text())="{p}"]')
                driver.execute_script("arguments[0].click();", page_btn)
                time.sleep(3)
                print(f"  → {p}페이지로 이동")
            except:
                print(f"  ⚠️ {p}페이지 이동 실패")
                break
    
    print(f"  ✅ {page_num}페이지 도착\n")

def click_store(driver, link_element):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", link_element)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", link_element)
    time.sleep(2)

def switch_to_entry_iframe(driver):
    driver.switch_to.default_content()
    WebDriverWait(driver, 10).until(
        EC.frame_to_be_available_and_switch_to_it((By.ID, "entryIframe"))
    )

def back_to_list(driver):
    driver.switch_to.default_content()
    driver.switch_to.frame("searchIframe")
    time.sleep(1.5)

def scroll_in_search_list(driver):
    """검색 결과 리스트를 천천히 스크롤"""
    scroll_container = driver.find_element(By.ID, '_pcmap_list_scroll_container')
    last_height = driver.execute_script("return arguments[0].scrollHeight", scroll_container)
    
    scroll_pause = 1.5
    max_scrolls = 30
    scroll_count = 0
    
    while scroll_count < max_scrolls:
        driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", scroll_container)
        time.sleep(scroll_pause)
        
        new_height = driver.execute_script("return arguments[0].scrollHeight", scroll_container)
        
        if new_height == last_height:
            break
            
        last_height = new_height
        scroll_count += 1
    
    driver.execute_script("arguments[0].scrollTo(0, 0);", scroll_container)
    time.sleep(1)

def click_tab(driver, tab_name):
    """탭 클릭"""
    try:
        tab = driver.find_element(By.XPATH, f'//a[@role="tab" and contains(., "{tab_name}")]')
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(2)
        return True
    except:
        try:
            tab = driver.find_element(By.XPATH, f'//a[@role="tab"]//span[text()="{tab_name}"]')
            driver.execute_script("arguments[0].click();", tab)
            time.sleep(2)
            return True
        except:
            return False

def crawl_page(page_num, collected_ids):
    """한 페이지 크롤링"""
    print(f"\n{'='*50}")
    print(f"  {page_num}페이지 크롤링 시작")
    print(f"{'='*50}\n")
    
    # 새 드라이버 생성
    driver = create_driver()
    page_store_data = []
    
    try:
        # 해당 페이지로 이동
        navigate_to_page(driver, page_num)
        
        # 스크롤
        scroll_in_search_list(driver)
        time.sleep(2)
        
        store_items = driver.find_elements(By.CSS_SELECTOR, '#_pcmap_list_scroll_container > ul > li')
        print(f"📋 현재 페이지 총 {len(store_items)}개 항목 발견\n")

        processed_count = 0
        for idx in range(len(store_items)):
            try:
                # iframe 초기화
                try:
                    driver.switch_to.default_content()
                    driver.switch_to.frame("searchIframe")
                except:
                    pass
                
                current_items = driver.find_elements(By.CSS_SELECTOR, '#_pcmap_list_scroll_container > ul > li')
                
                if idx >= len(current_items):
                    continue
                
                li = current_items[idx]

                # 기본 정보 추출
                try:
                    name_elem = li.find_element(By.CSS_SELECTOR, 'span.TYaxT')
                    name = name_elem.text.strip()
                    
                    category_elem = li.find_element(By.CSS_SELECTOR, 'span.KCMnt')
                    category = category_elem.text.strip()
                    
                    link_elem = li.find_element(By.CSS_SELECTOR, 'div.CHC5F div.bSoi3 a')
                    
                    store_id = f"{name}_{category}"
                        
                except Exception as e:
                    print(f"[{idx+1}/{len(store_items)}] ❌ 기본 정보 추출 실패 - {e}")
                    continue
                
                if store_id in collected_ids:
                    print(f"[{idx+1}/{len(store_items)}] 🔄 {name} - 중복, 스킵")
                    continue
                
                print(f"\n[{idx+1}/{len(store_items)}] 🏪 {name} 수집 중...")
                
                click_store(driver, link_elem)
                switch_to_entry_iframe(driver)
                
                # 상세 정보 수집
                address = ""
                contact = ""
                business_hours_text = ""
                summary = ""
                menus = []
                review_count = "0"
                reviews = []

                # 1. 주소
                try:
                    address = driver.find_element(By.CSS_SELECTOR, 'span.LDgIH').text.strip()
                    print(f"  ✅ 주소: {address}")
                except:
                    print(f"  ❌ 주소: 없음")
                
                time.sleep(1)

                # 2. 영업시간
                try:
                    time_blocks = driver.find_elements(By.CSS_SELECTOR, 'div.w9QyJ span.A_cdD')
                    
                    if not time_blocks:
                        try:
                            toggle_area = driver.find_element(By.CSS_SELECTOR, 'div.w9QyJ')
                            driver.execute_script("arguments[0].click();", toggle_area)
                            time.sleep(2)
                            time_blocks = driver.find_elements(By.CSS_SELECTOR, 'div.w9QyJ span.A_cdD')
                        except:
                            pass

                    business_hours = []
                    for b in time_blocks:
                        try:
                            day = b.find_element(By.CSS_SELECTOR, 'span.i8cJw').text
                            hours = b.find_element(By.CSS_SELECTOR, 'div.H3ua4').text
                            business_hours.append(f"{day} {hours}")
                            time.sleep(0.2)
                        except: 
                            pass

                    business_hours_text = " | ".join(business_hours)
                    business_hours_text = re.sub(r'[^\w\s:~\-|]', '', business_hours_text)

                    if business_hours_text:
                        print(f"  ✅ 영업시간: {business_hours_text[:50]}...")
                    else:
                        print(f"  ❌ 영업시간: 없음")
                except:
                    print(f"  ❌ 영업시간: 수집 실패")
                
                time.sleep(1)

                # 3. 연락처
                try:
                    contact = driver.find_element(By.CSS_SELECTOR, 'span.xlx7Q').text.strip()
                    print(f"  ✅ 연락처: {contact}")
                except:
                    print(f"  ❌ 연락처: 없음")
                
                time.sleep(1)

                # 4. 요약
                try:
                    summary_elem = None
                    try:
                        summary_elem = driver.find_element(By.CSS_SELECTOR, 'span.D6F_q')
                    except:
                        try:
                            summary_elem = driver.find_element(By.CSS_SELECTOR, 'span.yfzki')
                        except:
                            pass

                    if summary_elem:
                        summary = summary_elem.text.strip()
                        print(f"  ✅ 요약: {summary[:30]}...")
                    else:
                        print(f"  ❌ 요약: 없음")
                except:
                    print(f"  ❌ 요약: 수집 실패")
                
                time.sleep(1)

                # 5. 메뉴
                if click_tab(driver, "메뉴"):
                    print(f"  📋 메뉴 탭 클릭 완료")
                    
                    driver.execute_script("window.scrollTo(0, 300);")
                    time.sleep(1)
                    
                    try:
                        menu_names = driver.find_elements(By.CSS_SELECTOR, 'span.lPzHi')
                        menu_prices = driver.find_elements(By.CSS_SELECTOR, 'div.GXS1X em')
                        
                        total_menus = min(len(menu_names), len(menu_prices))
                        limit = total_menus if total_menus <= 3 else 3
                        
                        for i in range(limit):
                            try:
                                name = menu_names[i].text.strip()
                                price = menu_prices[i].text.strip()
                                if name and price:
                                    menus.append(f"{name}: {price}")
                            except:
                                pass

                        if menus:
                            print(f"  ✅ 메뉴: {len(menus)}개 수집 - 예: {menus[0]}")
                        else:
                            print(f"  ❌ 메뉴: 없음")
                    except:
                        print(f"  ❌ 메뉴: 수집 실패")
                else:
                    print(f"  ⚠️ 메뉴 탭 없음")
                
                time.sleep(1)

                # 6. 리뷰
                if click_tab(driver, "리뷰"):
                    print(f"  💬 리뷰 탭 클릭 완료")
                    
                    driver.execute_script("window.scrollTo(0, 300);")
                    time.sleep(1)
                    
                    # 리뷰 개수
                    try:
                        review_count = driver.find_element(By.CSS_SELECTOR, 'em.place_section_count').text.strip()
                        print(f"  ✅ 리뷰 개수: {review_count}")
                    except:
                        print(f"  ❌ 리뷰 개수: 없음")

                    # 리뷰 더보기
                    try:
                        more_btns = driver.find_elements(By.CSS_SELECTOR, 'span.TeItc')
                        if len(more_btns) >= 1:
                            driver.execute_script("arguments[0].click();", more_btns[0])
                            time.sleep(1.5)
                    except:
                        pass

                    # 리뷰 3개
                    try:
                        review_elements = driver.find_elements(By.CSS_SELECTOR, 'a[data-pui-click-code="rvshowmore"]')
                        for r in review_elements[:3]:
                            review_text = r.text.strip()
                            if review_text:
                                reviews.append(review_text)

                        if reviews:
                            print(f"  ✅ 리뷰: {len(reviews)}개 수집")
                        else:
                            print(f"  ❌ 리뷰: 없음")
                    except:
                        print(f"  ❌ 리뷰: 수집 실패")
                else:
                    print(f"  ⚠️ 리뷰 탭 없음")
                
                # 데이터 저장
                page_store_data.append({
                    "store_id": store_id,
                    "name": name,
                    "category": category,
                    "address": address,
                    "contact": contact,
                    "business_hours": business_hours_text,
                    "summary": summary,
                    "menus": menus,
                    "review_count": review_count,
                    "reviews": reviews
                })
                
                collected_ids.add(store_id)
                processed_count += 1
                
                print(f"  ✅ 수집 완료!")
                
                back_to_list(driver)
                
            except Exception as e:
                print(f"[{idx+1}] ❌ 오류: {str(e)[:100]}")
                try:
                    back_to_list(driver)
                except:
                    pass
        
        print(f"\n📊 {page_num}페이지 완료: {processed_count}개 수집")
        
        # 페이지 저장
        if page_store_data:
            df = pd.DataFrame(page_store_data)
            df.to_csv(f"busan_restaurants_page{page_num}.csv", index=False, encoding='utf-8-sig')
            print(f"💾 페이지 {page_num} 저장 완료\n")
        
    finally:
        # 드라이버 종료
        driver.quit()
        gc.collect()
        print(f"🔄 드라이버 종료 및 메모리 클리어\n")
    
    return page_store_data

# ===== 메인 실행 =====
if __name__ == "__main__":
    start_time = datetime.datetime.now()
    print(f"🚀 크롤링 시작: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    collected_ids = set()
    MAX_PAGES = 5  # ⭐ 5페이지
    
    for page in range(1, MAX_PAGES + 1):
        try:
            page_data = crawl_page(page, collected_ids)
            
            # 페이지 간 휴식 (봇 탐지 회피)
            if page < MAX_PAGES:
                print(f"😴 다음 페이지 전 30초 휴식...\n")
                time.sleep(30)
                
        except Exception as e:
            print(f"❌ {page}페이지 크롤링 실패: {e}")
            continue
    
    end_time = datetime.datetime.now()
    print(f"\n{'='*50}")
    print(f"  🎉 전체 크롤링 완료!")
    print(f"{'='*50}")
    print(f"시작: {start_time.strftime('%H:%M:%S')}")
    print(f"종료: {end_time.strftime('%H:%M:%S')}")
    print(f"소요: {end_time - start_time}")
    print(f"총 수집: {len(collected_ids)}개")

🚀 크롤링 시작: 2026-01-21 02:11:28


  1페이지 크롤링 시작

  ✅ 1페이지 도착

📋 현재 페이지 총 52개 항목 발견


[1/52] 🏪 중경식객 부산서면점 수집 중...
  ✅ 주소: 부산 부산진구 서전로10번길 39 1층 중경식객 부산서면점
  ✅ 영업시간: 수 11:00 - 23:00 | 목 11:00 - 23:00 | 금 11:00 - 24:0...
  ✅ 연락처: 051-809-7789
  ✅ 요약: 중경식객 부산서면점은 다양한 재료를 무한리필로 즐길 수...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 무한리필 훠궈(저녁,주말,공휴일): 19,800
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 505
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[2/52] 🏪 궁중해물탕 조씨집 대연본점 수집 중...
  ✅ 주소: 부산 남구 진남로 7-11 궁중해물탕 조씨집
  ❌ 영업시간: 없음
  ✅ 연락처: 0507-1323-4749
  ✅ 요약: 1959년부터 이어온 궁중해물탕 조씨집은 해물탕과 모듬...
  📋 메뉴 탭 클릭 완료
  ❌ 메뉴: 없음
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 663
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[3/52] 🏪 광안제일분식 수집 중...
  ✅ 주소: 부산 수영구 수영로540번길 77
  ✅ 영업시간: 매일 00:00 - 24:00...
  ✅ 연락처: 0507-1454-2291
  ✅ 요약: 광안제일분식은 분식뿐만 아니라 두루치기정식 등 다양한 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 쫀득감자볼: 4,200
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 373
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[4/52] 🏪 디저트 로지 수집 중...
  ✅ 주소: 부산 부산진구 서전로37번길 20 1층 디저트로지
  ✅ 영업시간: 수 12:00 - 21:00 | 목 12:00 - 21:00 | 금 12:00 - 21:0...
  ✅

  ✅ 영업시간: 수 11:20 - 21:00
15:00 - 17:20 브레이크타임
20:10 라스트오더 |...
  ✅ 연락처: 0507-1357-1653
  ✅ 요약: 기찰반점 중식뷔페는 다양한 중식 메뉴를 무한리필로 즐길...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 중식뷔페 무한리필 성인1인 (런치): 12,900
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 801
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[30/52] 🏪 부산당 해운대본점 수집 중...
  ✅ 주소: 부산 해운대구 해운대로570번길 45 1층, 2층
  ✅ 영업시간: 매일 08:00 - 21:00...
  ✅ 연락처: 0507-1483-9594
  ✅ 요약: 부산당 해운대본점은 가성비 좋은 빵집으로, 단팥빵, 소...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 딸기가 막히네: 68,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 83
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[31/52] 🏪 류센소 본점 수집 중...
  ✅ 주소: 부산 해운대구 구남로8번길 62 1F
  ✅ 영업시간: 매일 11:00 - 21:30
21:00 라스트오더...
  ✅ 연락처: 0507-1493-8283
  ✅ 요약: 류센소 본점은 해운대 부민병원 인근에 위치한 라멘 맛집...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 류센소: 11,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 3,751
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[32/52] 🏪 해운대 하선집 숯불돼지갈비 전문점 수집 중...
  ✅ 주소: 부산 해운대구 좌동순환로 395 1층 105, 106, 107, 108호
  ❌ 영업시간: 없음
  ✅ 연락처: 0507-1340-1139
  ✅ 요약: 해운대 하선집은 숯불향이 가득한 돼지갈비와 신선한 한우...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 48개월 이하 어린이도시락+미역국: 27

  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 몽벨 프렌치토스트: 10,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 5,954
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[8/60] 🏪 푸주인 서면점 수집 중...
  ✅ 주소: 부산 부산진구 가야대로784번길 46-11 1층 101호
  ✅ 영업시간: 수 11:00 - 24:00
23:30 라스트오더 | 목 11:00 - 24:00
23:3...
  ✅ 연락처: 0507-1410-2888
  ❌ 요약: 없음
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: ❤️늦은밤, 식사류 1개 공짜❤️: 15,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 812
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[9/60] 🏪 광안다찌 광안리본점 수집 중...
  ✅ 주소: 부산 수영구 광안해변로344번길 17-20 산바다회타운 6층
  ❌ 영업시간: 없음
  ✅ 연락처: 0507-1362-0388
  ✅ 요약: 광안다찌는 광안리해수욕장 근처에 위치한 횟집으로, 신선...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 2개 수집 - 예: 다찌 한상(1인): 55,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 4,621
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[10/60] 🏪 LAB 이흥용과자점 두실점 수집 중...
  ✅ 주소: 부산 금정구 금단로 80 LAB 이흥용과자점
  ✅ 영업시간: 매일 08:30 - 21:00...
  ✅ 연락처: 0507-1427-1998
  ✅ 요약: 이흥용과자점 두실점은 넓은 매장과 다양한 빵 종류로 주...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 1개 수집 - 예: 저염숙성명란바게트: 4,200
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 95
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[11/60] 🏪 바오하우스 광안점 수집 중...
  ✅ 주소: 부산 수영구 광안로48번길 5 1층
  ✅ 영업시간: 수 11:00 - 21:00


  ❌ 메뉴: 없음
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 452
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[38/60] 🏪 소각 대연점 수집 중...
  ✅ 주소: 부산 남구 유엔평화로 10 1층
  ✅ 영업시간: 매일 11:30 - 24:00
23:00 라스트오더...
  ✅ 연락처: 0507-1319-4616
  ✅ 요약: 소각 대연점은 다양한 부위의 소고기를 가성비 좋게 즐길...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 매일 직접 끓여 준비하는 소고기무국: 9,800
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 3,369
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[39/60] 🏪 연막창 동래점 수집 중...
  ✅ 주소: 부산 동래구 명륜로129번나길 16 1층
  ✅ 영업시간: 매일 15:00 - 02:00
01:00 라스트오더...
  ✅ 연락처: 0507-1379-9292
  ✅ 요약: 연막창 동래점은 대구에서 유명한 막창 전문점으로, 동래...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 초벌연막창 200g ★★: 12,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 361
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[40/60] 🏪 빨간등대 영도본점 수집 중...
  ✅ 주소: 부산 영도구 남항서로 40 2층 빨간등대 영도본점
  ✅ 영업시간: 수 11:30 - 21:30
15:00 - 17:00 브레이크타임
20:00 라스트오더 |...
  ✅ 연락처: 0507-1382-7941
  ✅ 요약: 빨간등대 영도본점은 신선한 해산물과 아름다운 오션뷰로 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 대광어 코스요리: 55,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 2,981
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[41/60] 🏪 부산꼼장어맛집성일집 수집 중...
  ✅ 주소: 부산 중구 대교로 103 부산꼼장어맛집성일집
  ❌ 영업시간: 없음



[6/60] 🏪 1969부원동칼국수 남포본점 수집 중...
  ✅ 주소: 부산 중구 구덕로22번길 3 1층
  ✅ 영업시간: 수 10:50 - 19:30
15:00 - 17:00 브레이크타임
19:00 라스트오더 |...
  ✅ 연락처: 0507-1477-9925
  ✅ 요약: 1969부원칼국수 남포본점은 남포역 1번 출구 근처에 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 부원칼국수[자가제면]: 9,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,560
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[7/60] 🏪 금수복국 롯데백화점 부산본점 수집 중...
  ✅ 주소: 부산 부산진구 가야대로 772 롯데백화점 부산본점 9층 식당가
  ✅ 영업시간: 매일 10:30 - 20:30
20:00 라스트오더...
  ✅ 연락처: 0507-1423-3977
  ❌ 요약: 없음
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 미쉐린 특별정식: 29,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 829
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[8/60] 🏪 일백풍천민물장어 명지직영점 수집 중...
  ✅ 주소: 부산 강서구 르노삼성대로 638-10 2층
  ✅ 영업시간: 매일 11:30 - 22:00...
  ✅ 연락처: 0507-1353-9176
  ✅ 요약: 일백풍천민물장어 명지직영점은 오픈한 지 얼마 되지 않은...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 풍천민물장어[국내산최상등급]: 58,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 240
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[9/60] 🏪 트로피 부산 수집 중...
  ✅ 주소: 부산 동구 자성로 119 2층
  ✅ 영업시간: 매일 11:00 - 22:00
14:30 - 16:30 브레이크타임
21:00 라스트오더...
  ✅ 연락처: 0507-1430-1311
  ✅ 요약: 트로피 부산은 한우육회와 철판샤브를 

  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,752
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[36/60] 🏪 옹드 수집 중...
  ✅ 주소: 부산 해운대구 좌동순환로 443 102호
  ✅ 영업시간: 수 12:00 - 19:00 | 목 12:00 - 19:00 | 금 12:00 - 19:0...
  ✅ 연락처: 0507-1389-8062
  ✅ 요약: 옹드는 부산 해운대의 숨은 디저트 맛집입니다....
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 두바이 쫀득쿠키: 5,300
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 112
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[37/60] 🏪 후라토식당 서면점 수집 중...
  ✅ 주소: 부산 부산진구 서면로 65-4 지상 1층
  ✅ 영업시간: 매일 11:30 - 21:50
14:50 - 17:00 브레이크타임
20:50 라스트오더...
  ✅ 연락처: 051-997-2046
  ❌ 요약: 없음
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 규카츠: 19,500
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 4,064
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[38/60] 🏪 부산선옥 수집 중...
  ✅ 주소: 부산 수영구 민락로27번길 27 1층 101호
  ✅ 영업시간: 수 17:00 - 24:00
23:00 라스트오더 | 목 17:00 - 24:00
23:0...
  ✅ 연락처: 0507-1466-9309
  ✅ 요약: 부산선옥은 지리산 흑돼지 오겹살을 맛볼 수 있는 고기 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 지리산 흑돼지 오겹살 한판(440g): 48,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,315
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[39/60] 🏪 배비장보쌈구서본점 수집 중...
  ✅ 주소: 부산 금정구 금정로 239 배비장보쌈 구서본점
  ✅ 영업시간: 수 11:00 - 22:00 | 목 

  ✅ 요약: 박복순 쪽갈비 연산동본점은 연산역 근처에 위치한 인기 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: <SET> A세트 (2-3인): 59,900
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 3,244
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[4/60] 🏪 메츠과자점 본점 수집 중...
  ✅ 주소: 부산 남구 수영로266번길 46
  ✅ 영업시간: 수 08:00 - 22:00 | 목 08:00 - 22:00 | 금 08:00 - 22:0...
  ✅ 연락처: 0507-1405-4546
  ✅ 요약: 메츠과자점 본점은 다양한 빵과 디저트를 즐길 수 있는 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 앙버터소금빵: 3,200
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,180
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[5/60] 🏪 빡돈 통쪽갈비 동래점 수집 중...
  ✅ 주소: 부산 동래구 충렬대로179번길 27-5 1층 빡돈 통쪽갈비 동래점
  ✅ 영업시간: 수 17:00 - 02:00
00:00 라스트오더 | 목 17:00 - 02:00
00:0...
  ✅ 연락처: 0507-1369-7909
  ✅ 요약: 빡돈 통쪽갈비 동래점은 동래역에서 도보 2분 거리에 있...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 세쪽세트: 86,900
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 544
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[6/60] 🏪 야키토리백탄 서면점 수집 중...
  ✅ 주소: 부산 부산진구 부전로 75-3 1층
  ✅ 영업시간: 수 17:30 - 01:00
00:00 라스트오더 | 목 17:30 - 01:00
00:0...
  ✅ 연락처: 0507-1320-9201
  ✅ 요약: 야키토리백탄 서면점은 광안점에 이어 오픈한 인기 이자카...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 백 코스: 38,000
  💬 리뷰 탭 클릭 완

  ✅ 리뷰 개수: 55
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[34/60] 🏪 파스타빌라 수집 중...
  ✅ 주소: 부산 부산진구 동성로25번길 7
  ✅ 영업시간: 수 11:00 - 20:50
15:00 - 16:00 브레이크타임
20:00 라스트오더 |...
  ✅ 연락처: 051-805-0777
  ✅ 요약: 파스타빌라는 서면에서 손꼽히는 파스타 맛집으로, 트러플...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 채끝살스테이크: 39,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 3,967
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[35/60] 🏪 광안리 숯불생선구이 이공이 수집 중...
  ✅ 주소: 부산 수영구 광안해변로 177 3층
  ✅ 영업시간: 매일 06:00 - 23:00...
  ✅ 연락처: 0507-1331-4716
  ✅ 요약: 광안리 숯불생선구이 이공이는 숯불로 구운 생선 소금구이...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 2개 수집 - 예: 숯불 생선 소금구이: 23,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 951
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[36/60] 🏪 부엌간 해운대점 수집 중...
  ✅ 주소: 부산 해운대구 달맞이길117번나길 200 2층
  ✅ 영업시간: 매일 11:00 - 21:00
15:00 - 16:30 브레이크타임
20:30 라스트오더...
  ✅ 연락처: 0507-1362-9023
  ✅ 요약: 부엌간 해운대점은 해운대 달맞이길에 위치한 분위기 좋은...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 엄마들이 가장 좋아하는 레스토랑: 8,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 2,312
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[37/60] 🏪 미죠떼 수집 중...
  ✅ 주소: 부산 해운대구 달맞이길117번가길 222 모닝힐 206호 미죠떼
  ✅ 영업시간: 수 11:00 - 22:00
15:30 - 1

  ✅ 요약: 물푸레는 기장 아난티 근처에 위치한 분위기 좋은 레스토...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 숏립 짚불 BBQ & 대파구이: 108,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 144
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[3/20] 🏪 웨이양 수집 중...
  ✅ 주소: 부산 남구 용소로7번길 78
  ✅ 영업시간: 수 17:00 - 24:00
23:00 라스트오더 | 목 17:00 - 24:00
23:0...
  ✅ 연락처: 0507-1410-1744
  ✅ 요약: 웨이양은 양고기 요리와 다양한 중국식 메뉴를 맛볼 수 ...
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 예약필수) 숯불 양다리구이 3인: 75,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,606
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[4/20] 🏪 사미헌 수집 중...
  ✅ 주소: 부산 부산진구 서면문화로 19
  ✅ 영업시간: 수 11:00 - 21:30
20:30 라스트오더 | 목 11:00 - 21:30
20:3...
  ✅ 연락처: 051-819-6677
  ✅ 요약: 사미헌은 부산 서면에서 유명한 한우 전문점입니다....
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 한우 상강꽃살: 57,000
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 1,623
  ✅ 리뷰: 3개 수집
  ✅ 수집 완료!

[5/20] 🏪 코지하우스 동래점 수집 중...
  ✅ 주소: 부산 동래구 온천천로 165 동래롯데캐슬퀸 102동 201호, 202호
  ✅ 영업시간: 수 11:30 - 21:30
15:30 - 17:00 브레이크타임
14:40 20:40 라...
  ✅ 연락처: 051-556-1915
  ❌ 요약: 없음
  📋 메뉴 탭 클릭 완료
  ✅ 메뉴: 3개 수집 - 예: 10$ 스테이크: 13,900
  💬 리뷰 탭 클릭 완료
  ✅ 리뷰 개수: 3,416
  ✅ 리뷰: 3개 수집
 

In [2]:
driver.quit()

NameError: name 'driver' is not defined